# DONUT SPRINKLES - Bias/Variance Lab

## Where This Fits

The model selection lab gave us a practical recipe to trade off underfitting against overfitting: hold back a test set, use cross validation on the training data to choose a hyperparameter, and open the test set exactly once to provide a final performance estimate.

This lab introduces a mathematical framework that allows us to quantify the notions of underfitting and overfitting. 

**Enter all group names in the cell below:**

YOUR ANSWER HERE

## Your First Day at DonutCorp

Welcome to your first day on the job as a data analyst at DonutCorp! Your first task is to develop a model that can predict donut sales as a function of chocolate sprinkle density.  You will work with a data set that has been carefully gathered by diligent DonutCorp employees.  Execute the cell below to import the required libraries, load the data, and visualize it...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sprinkle_data
import fit_experiments

plt.rcParams['figure.figsize'] = [4.5, 3.5]
plt.rcParams['font.size'] = 12
plt.rcParams['lines.markersize'] = 4

sprinkles = np.loadtxt('data/sprinkles.csv', delimiter=',')
x = sprinkles[:, 0]
y = sprinkles[:, 1]
plt.plot(x, y, 'o')
plt.xlabel('sprinkle density')
plt.ylabel('sales')
plt.show()

## Fit a Line
As a first pass, let's fit a line to the data.  That line can then be used to make predictions about the impact of sprinkle density choices...

In [ ]:
plt.plot(x, y, 'o')
model = fit_experiments.PolynomialRegression()
model.fit(x, y, degree=1)
model.plot()
plt.xlabel('sprinkle density')
plt.ylabel('sales')
plt.show()

## Aside... What does it actually *mean* to "fit a line to data"?
In what sense is the line above the "best fit" for this data?  It is the line that minimizes the mean squared prediction error or MSE:

$$\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2$$

This is the average squared error between the observed output $y_i$ and the predicted output $\hat{y}_i$.  In other words, we are selecting the line that has the lowest possible average square length for the red lines in the figure below...

In [ ]:
plt.plot(x, y, 'o')
model.fit(x, y, degree=1)
model.plot(x, y, show_error=True)
mse = model.evaluate(x, y)
plt.xlabel('sprinkle density')
plt.ylabel('sales')
print("The MSE for this line is: {:.2f}".format(mse))
plt.show()

## SOME QUESTIONS...
* According to this model, what is the approximate predicted sales for a sprinkle concentration of .7?
* Do you think this is a good prediction?  What value would you predict based on eyeballing the training data?
* Any ideas on how we might improve this model? 

YOUR ANSWER HERE

## Fitting a Polynomial
There is no need to limit ourselves to fitting a straight line to this data.  We can just as easily fit a polynomial of any degree.  Try changing `MY_DEGREE` in the cell below and re-running it.  As you increase the degree of the polynomial, you should see a progressively closer fit to the data points. Go ahead and settle on a final model by picking a polynomial degree that you think will be most effective at predicting future sales.

In [ ]:
# Try several values here.  Once you have settled on a final choice, leave
# MY_DEGREE set to that value and make sure this cell has been run: the rest
# of the notebook uses whatever model this cell produced.
MY_DEGREE = 1

model.fit(x, y, degree=MY_DEGREE)
plt.plot(x, y, 'o')
model.plot()
plt.xlabel('sprinkle density')
plt.ylabel('sales')
print("training MSE: {:.3f}".format(model.evaluate(x, y)))
plt.show()

---

## Judgement Day

OK!  You've sent your model out into the wild.  It is being used in DonutCorp stores all over America!  Now a second set of data arrives from your intrepid data collectors. Execute the cell below to see the new set of data.  Notice that it looks a bit different from the data you have been working with.

In [ ]:
sprinkles_tst = np.loadtxt('data/sprinkles_test.csv', delimiter=',')
x_tst = sprinkles_tst[:, 0]
y_tst = sprinkles_tst[:, 1]
plt.plot(x_tst, y_tst, 'o')
plt.xlabel('sprinkle density')
plt.ylabel('sales')
plt.show()

## QUESTIONS:

* Is it surprising that this data looks different? Why might it look different?

YOUR ANSWER HERE

Let's go ahead and see how your model does on this new set of data... 

In [ ]:
plt.plot(x_tst, y_tst, 'o')
model.plot(x_tst, y_tst, show_error=True)
plt.xlabel('sprinkle density')
plt.ylabel('sales')

mse = model.evaluate(x_tst, y_tst)
print("The MSE for this line is: {:.2f}".format(mse))
plt.show()

## QUESTIONS:

* Are you satisfied with the performance of your model?
* You were asked to pick the degree by eyeballing the resulting fit.  Your boss probably won't be impressed with this methodology.  Describe a more principled approach.

YOUR ANSWER HERE

## WARNING We Venture Now Into The Realm of the Thought Experiment

Let's assume that an infinitely intelligent space alien beams in and shows us the TRUE relationship between sprinkle concentrations and sales. It turns out that sprinkle sales are governed by the formula
$$ y = f(x) + \epsilon$$

where $\epsilon$ is a random variable drawn from the normal distribution $\mathcal{N}(0, .07)$.  The function $f(x)$ captures the "true" relationship between sprinkles and sales, while the $\epsilon$ term captures the fact that sales are inherently unpredictable if all we are working from is the concentration of sprinkles.

It is a common assumption in statistical machine learning that our data source has this form: some "true" underlying function corrupted by random noise.  Execute the cell below to see $f(x)$.


In [ ]:
ds = sprinkle_data.SprinkleDataSource()
xs = np.arange(0, 1, .01)
ys = ds.true_fun(xs)
plt.plot(xs, ys)
plt.xlabel('sprinkle density')
plt.ylabel('expected sales')
plt.show()

In the real world we *never actually have access to $f(x)$*.  Instead, we get noisy data drawn from some unknown underlying distribution.  The goal is to minimize the *expected* squared error of our fit: $$E[(y - \hat{f}(x))^2]$$
Where $\hat{f}(x)$ is our estimate, $y$ is the true target value, and the expectation is taken over the choice of training set... In other words this describes the average error we would expect to see if we repeatedly pulled data sets from our underlying distribution, fit a model, then used that model to make predictions.

We won't do the derivation here, but it turns out that expected error can be broken down into three components: 
$$E[(y - \hat{f}(x))^2] = (Bias[\hat{f}(x)])^2 + Variance[\hat{f}(x)] + \sigma^2$$

where

$Bias[\hat{f}(x)] = E[\hat{f}(x)] - f(x)$,

$Variance[\hat{f}(x)] = E[(\hat{f}(x) - E[\hat{f}(x)])^2]$,

and

$\sigma^2$ is the variance of the noise term


To make this concrete, let's look at what happens when we repeatedly fit a polynomial to different data sets drawn from the same underlying (sprinkle) source...


In [ ]:
mean_sqrd_bias, mean_var = fit_experiments.bias_variance_experiment(num_trials=100,
                                                                    train_size=30,
                                                                    degree=3,
                                                                    source=ds, display=True)

The many blue lines in this figure represent many possible least-squares fits, each one for a different data set drawn from the true distribution.  The light green line represents the average of all of these fits: $E[\hat{f}(x)]$.  The red line represents the true data function $f(x)$.  Referring back to our bias variance decomposition, expected error is the sum of three sources: 
1. Squared bias, $(E[\hat{f}(x)] - f(x))^2$, is related to the difference between the light green line and the red line.  If we have a large squared bias, that means that our average model is bad, even when it is averaged across many attempts.
2. Variance $E[(\hat{f}(x) - E[\hat{f}(x)])^2]$ is a measure of how spread out our individual models are relative to our average model.  Large variance means that we learn a radically different model depending on which data set we happen to learn from.
3. Irreducible error $\sigma^2$ comes from the fact that the system we are trying to model has an inherently random component that is fundamentally un-learnable.

Since we can't do anything about irreducible error, our goal is to minimize both bias and variance.  Unfortunately, decreasing bias tends to increase variance and vice-versa. This is often described as the bias/variance dilemma and is a central challenge in machine learning.  Go ahead and re-run the experiment in the cell above a few times with different degrees for the polynomial fit (third argument).  

QUESTION: 
* How does changing the degree of the polynomial impact the bias and variance values? 


YOUR ANSWER HERE

---

Now let's try systematically evaluating the bias and variance as we modify the degree of the polynomial...

In [ ]:
# Note that max_degree is exclusive: this covers degrees 0 through 7.
fit_experiments.tune_experiment(num_trials=10000, train_size=30, 
                                min_degree=0, 
                                max_degree=8, source=ds)

## QUESTIONS:
* Do these results match your intuitions?
* Given these results, what degree polynomial should we have used for our original learning problem?
* Try re-running the experiment above with larger and smaller data sets.  How does changing the size of the training set impact the bias and variance?
* Two common problems in machine learning are **overfitting** or **underfitting**.  Overfitting means that our trained model includes irrelevant details from the training data and does not generalize well.  Underfitting means that our model is ignoring structure in the data that could be used to make better predictions. Explain how these ideas relate to the bias/variance decomposition.
* This experiment required an infinitely intelligent space alien.  What exactly did we need the alien for?

YOUR ANSWER HERE

---

## So How Do We Do This Without the Alien?

The experiment above is not something you could ever run at DonutCorp.  Bias is defined as a distance from $f(x)$, and nobody has $f(x)$.  If we did, we would not need a model at all.

Here is the point of this whole lab.  You already know a procedure that does not need the alien.  Cross validation estimates expected error directly from the data you already have, without ever separating it into bias and variance.  You do not need to write any of it here: the cell below runs it on the same 30 training points and plots it against the decomposition you just computed.

Note that the decomposition column now includes $\sigma^2$, so that both columns are estimates of the same thing: the total expected squared error.

In [ ]:
from sklearn.model_selection import RepeatedKFold

# tune_experiment plots degrees range(min_degree, max_degree), so the cell
# above covered 0 through 7.  Match that here.
degrees = range(0, 8)

# The variance of the noise term, from the formula the alien gave us.
NOISE_VARIANCE = 0.07

# The decomposition.  This needs the alien: it draws 10000 fresh training
# sets and compares every fit against the true f(x).
predicted = []
for degree in degrees:
    sqrd_bias, var = fit_experiments.bias_variance_experiment(
        num_trials=10000, train_size=30, degree=degree, source=ds,
        display=False)
    predicted.append(sqrd_bias + var + NOISE_VARIANCE)

# Cross validation.  This needs nothing but the 30 points we started with.
rkf = RepeatedKFold(n_splits=10, n_repeats=20, random_state=0)
cv_scores = []
for degree in degrees:
    fold_errors = []
    for train_index, val_index in rkf.split(x):
        fit = np.poly1d(np.polyfit(x[train_index], y[train_index], degree))
        fold_errors.append(np.mean((y[val_index] - fit(x[val_index]))**2))
    cv_scores.append(np.mean(fold_errors))

print("{:>7} {:>26} {:>20}".format("", "bias^2 + variance + sigma^2", "cross validation"))
print("{:>7} {:>26} {:>20}".format("degree", "(needs the alien)", "(needs only data)"))
for degree, pred, cv in zip(degrees, predicted, cv_scores):
    print("{:>7} {:>26.4f} {:>20.4f}".format(degree, pred, cv))

print("\nlowest predicted error   : degree {}".format(
    min(zip(predicted, degrees))[1]))
print("lowest cross validation  : degree {}".format(
    min(zip(cv_scores, degrees))[1]))

plt.plot(list(degrees), predicted, 'o-',
         label=r'$bias^2$ + variance + $\sigma^2$')
plt.plot(list(degrees), cv_scores, 's-', label='cross validation')
plt.xlabel('polynomial degree')
plt.ylabel('expected squared error')
plt.legend(fontsize=9)
plt.show()

## QUESTIONS:

* Compare the two columns.  They do not agree on every number, but do they agree about which degrees are bad and roughly where the good ones are?
* Cross validation never computes a bias or a variance.  So what is the quantity it *is* estimating?  Look back at the decomposition formula.
* Why can cross validation get away with never separating bias from variance?
* We used all 30 of our available data points for this cross validation analysis, and it worked beautifully: it tuned the model about as well as the alien's infinite-data experiment did.  Did using all of the data for tuning cost us anything?

YOUR ANSWER HERE